<a href="https://colab.research.google.com/github/dvijiramesh/genai-ml-experiments/blob/main/rag-document-search/rag_document_search_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation (RAG) Document Search Demo

## Context

This notebook was developed as part of my ongoing learning in Generative AI systems.
The goal is to explore and implement a Retrieval-Augmented Generation (RAG) pipeline for document-based question answering.

## Goal

Build a simple RAG system that can answer questions by retrieving relevant information from a document and providing that context to a language model.

## Why RAG?

Large Language Models (LLMs) do not have access to private documents or newly created information.
RAG enables systems to retrieve relevant knowledge from external sources and use that information to generate accurate responses.

## Pipeline Overview

1. Load a source document
2. Split the document into smaller chunks
3. Convert text chunks into embeddings
4. Store embeddings in a vector database
5. Retrieve relevant chunks for a user query
6. Use the retrieved context to generate an answer

## Tools Used

* Python
* LangChain
* HuggingFace Embeddings
* FAISS Vector Database
* Google Colab


### **Step 1: Install and import the dependencies**  
- Import the necessary modules for document loading, embedding generation, vector storage, and pipeline setup

In [6]:
# pip install langchain_community

In [ ]:
# pip install faiss-cpu

In [ ]:
# pip install -U bitsandbytes

In [7]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import HuggingFacePipeline
import warnings
warnings.filterwarnings('ignore')


### **Step 2: Load the document**







In [8]:
from google.colab import files
files.upload()

Saving state_of_union.txt to state_of_union (1).txt


{'state_of_union (1).txt': b'Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \r\n\r\nLast year COVID-19 kept us apart. This year we are finally together again. \r\n\r\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \r\n\r\nWith a duty to one another to the American people to the Constitution. \r\n\r\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \r\n\r\nSix days ago, Russia\xe2\x80\x99s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \r\n\r\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \r\n\r\nHe met the Ukrainian people. \r\n\r\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determi

In [9]:
# text_loader = TextLoader("state_of_union.txt")
text_loader = TextLoader("state_of_union.txt")

In [10]:
# Convert file into LangChain document objects
text_document = text_loader.load()

# Preview first 200 characters
print(text_document[0].page_content[:200])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. 


- The document will later be chunked, embedded, and used to retrieve relevant context during the question-answering phase.

### **Step 3: Split the document into chunks**


In [11]:
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
split_texts = doc_splitter.split_documents(text_document)
print(len(split_texts))  # Prints the number of chunks the PDF has been split into


53


### **Step 4: Generate embeddings for each chunk**


In [12]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
hf_embed = HuggingFaceEmbeddings(model_name=MODEL_NAME)
text = split_texts[0].page_content
hf_embed_result = hf_embed.embed_documents([text])
print(len(hf_embed_result[0]))  # Prints the length of the first embedded document

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


768


In [13]:
embedded_chunks = [hf_embed.embed_query(chunk.page_content) for chunk in split_texts]

In [14]:
import pandas as pd
df_chunks = pd.DataFrame(embedded_chunks)
df_chunks


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.017570,0.101777,-0.010494,0.024629,0.007867,-0.008709,-0.047972,-0.011033,-0.000209,-0.036135,...,-0.019824,-0.019930,0.044173,0.027855,-0.045804,-0.009089,-0.035914,0.000358,-0.003852,-0.017099
1,0.033408,0.078639,-0.002185,-0.031801,-0.012582,-0.014186,-0.000025,-0.041827,0.026004,0.033043,...,-0.006442,-0.016889,-0.027715,0.053919,0.027535,0.008685,-0.011436,0.068335,-0.028501,-0.037242
2,0.012558,0.033666,0.001252,0.005224,-0.022721,-0.028479,0.052828,-0.022878,-0.009452,0.001424,...,-0.012291,-0.024313,0.042918,0.025881,0.014226,-0.048886,-0.016243,0.035996,-0.012137,-0.024365
3,-0.038429,0.036189,0.013976,0.010168,0.009914,-0.025288,-0.008883,-0.049233,0.026862,0.001242,...,-0.000914,-0.014002,0.074753,0.019576,0.037865,-0.024706,-0.000002,0.058000,-0.010562,0.000267
4,0.014516,0.041241,0.003573,-0.037763,-0.010100,-0.032179,0.032072,-0.017779,0.000895,-0.007063,...,-0.027593,0.015847,0.057699,0.030875,0.027842,0.000082,0.009793,-0.000430,-0.051848,-0.034995
5,0.041825,0.035288,0.004461,-0.040438,-0.023730,-0.059049,0.025266,-0.003494,0.019276,0.017754,...,-0.069936,0.001590,-0.001447,0.068728,0.011128,-0.009241,-0.027915,0.030503,-0.057551,-0.032336
6,0.021648,0.014052,0.023061,-0.040130,-0.015390,-0.032677,0.025847,-0.002982,0.084222,-0.001781,...,-0.043137,-0.005948,0.100516,0.046437,0.033617,-0.013371,0.003821,0.024070,-0.060905,-0.036954
7,0.000521,0.041569,0.005422,-0.008324,-0.044695,-0.049875,0.010778,-0.030330,0.007619,0.015853,...,-0.063684,0.022584,0.050347,0.042918,0.029160,-0.046929,0.000421,0.000400,-0.094893,-0.003460
8,0.032507,0.082838,0.012870,0.017960,-0.045969,-0.029158,0.019323,-0.018328,0.011206,0.019157,...,-0.043993,0.004471,0.044040,0.051866,0.042437,-0.076030,-0.006913,0.022640,-0.067280,-0.022953
9,-0.012315,0.038408,-0.010831,0.002273,0.016610,-0.036781,-0.011865,-0.025149,0.021735,-0.024317,...,-0.047840,0.013128,0.036254,0.035082,0.044569,-0.025483,0.009615,0.030583,-0.037560,-0.012570


### **Step 5: Build the FAISS vector store and create a retriever**


In [15]:
vectorstore=FAISS.from_documents(split_texts, hf_embed)

# It will take thesame embedding of the chunks as shown above and and crate a vecor database for it which will be temporary, ie non persistent

In [16]:
retriever=vectorstore.as_retriever()

In [17]:
# The way the retriever works

query = "What are the key points from the State Of The Union"
docs = retriever.invoke(query)
for doc in docs:
    print(doc.page_content)

Let’s increase Pell Grants and increase our historic support of HBCUs, and invest in what Jill—our First Lady who teaches full-time—calls America’s best-kept secret: community colleges. 

And let’s pass the PRO Act when a majority of workers want to form a union—they shouldn’t be stopped.  

When we invest in our workers, when we build the economy from the bottom up and the middle out together, we can do something we haven’t done in a long time: build a better America.
Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny.
But in my administration, the watch

In [18]:
query2 = "How is the United States supporting Ukraine economically and militarily?"

In [19]:
docs = retriever.invoke(query2)
for doc in docs:
    print(doc.page_content)

The Russian stock market has lost 40% of its value and trading remains suspended. Russia’s economy is reeling and Putin alone is to blame. 

Together with our allies we are providing support to the Ukrainians in their fight for freedom. Military assistance. Economic assistance. Humanitarian assistance. 

We are giving more than $1 Billion in direct assistance to Ukraine. 

And we will continue to aid the Ukrainian people as they defend their country and to help ease their suffering.
Along with twenty-seven members of the European Union including France, Germany, Italy, as well as countries like the United Kingdom, Canada, Japan, Korea, Australia, New Zealand, and many others, even Switzerland. 

We are inflicting pain on Russia and supporting the people of Ukraine. Putin is now isolated from the world more than ever. 

Together with our allies –we are right now enforcing powerful economic sanctions.
The United States is a member along with 29 other nations. 

It matters. American diplo

### **Step 6: Design a prompt template for the language model**


In [20]:
from langchain_core.prompts import ChatPromptTemplate

In [21]:
template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use one sentence and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [22]:
prompt=ChatPromptTemplate.from_template(template)

In [23]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [24]:
output_parser=StrOutputParser()

### **Step 7: Load and configure a quantized language model**


In [25]:
# pip install -U bitsandbytes

In [26]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model.eval()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear4bit(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear4bit(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=

In [27]:
model.eval()
generation_config = model.generation_config
# Set temperature to 0 for deterministic responses
generation_config.temperature = 0.8
# Set number of returned sequences to 1
generation_config.num_return_sequences = 1
# Set maximum new tokens per response
generation_config.max_new_tokens = 256
# Disable token caching
generation_config.use_cache = False
# Set repetition penalty for more diverse responses
generation_config.repetition_penalty = 1.7
# Define pad and EOS token IDs
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

### **Step 8: Set up the generation pipeline and chain the components**


In [28]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
    pipeline,
)

In [29]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Create the HuggingFacePipeline object
llm_pipeline = HuggingFacePipeline(pipeline=pipe)

In [30]:
rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_pipeline
    | output_parser
)

- The chain executes all RAG steps in sequence starting from retrieve, prompt, generate, and parse, producing a complete answer pipeline.

### **Step 9: Invoke the pipeline with a query**


In [31]:
result = rag_chain.invoke("How is the United States supporting Ukraine economically and militarily?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [32]:
result

'Human: You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don\'t know the answer, just say that you don\'t know.\nUse one sentence and keep the answer concise.\nQuestion: How is the United States supporting Ukraine economically and militarily?\nContext: [Document(id=\'dd68c2ab-e285-4978-9bb6-adb8477c9647\', metadata={\'source\': \'state_of_union.txt\'}, page_content=\'The Russian stock market has lost 40% of its value and trading remains suspended. Russia’s economy is reeling and Putin alone is to blame. \\n\\nTogether with our allies we are providing support to the Ukrainians in their fight for freedom. Military assistance. Economic assistance. Humanitarian assistance. \\n\\nWe are giving more than $1 Billion in direct assistance to Ukraine. \\n\\nAnd we will continue to aid the Ukrainian people as they defend their country and to help ease their suffering.\'), Document(id=\'18b2bc2c-9836-4a45-a041-56169ec